# Traffic-Sign Recognition — Results Showcase

This notebook is a **presentation layer**, not a training script. All heavy
lifting lives in the `traffic_signs` package; the notebook loads the
already-trained checkpoints from `checkpoints/` and the evaluation
artefacts from `reports/`, and renders them.

To reproduce the training runs, use the CLI instead:

```bash
python scripts/train_all.py
```

## Contents
1. Dataset overview (class distribution, sample grid)
2. Training curves per model
3. Confusion matrices and per-class reports
4. Comparison table
5. Interactive inference on a single image

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the src/ layout importable when running the notebook from notebooks/
ROOT = Path().resolve()
if (ROOT.parent / 'src').is_dir() and str(ROOT.parent / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT.parent / 'src'))
elif (ROOT / 'src').is_dir() and str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from traffic_signs.utils.logging_setup import configure_logging
configure_logging('INFO')

## 1. Dataset overview

A random sample per class plus the raw class-count distribution. Requires
the GTSRB training folder to be present — see `data/README.md`.

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

TRAIN_DIR = Path('../data/raw/GTSRB/Final_Training/Images')
assert TRAIN_DIR.is_dir(), f'Adjust TRAIN_DIR — {TRAIN_DIR} not found.'

class_dirs = sorted(p for p in TRAIN_DIR.iterdir() if p.is_dir())
print(f'Found {len(class_dirs)} classes.')

# Grid of one random image per class
cols = 8
rows = (len(class_dirs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 2.2))
rng = random.Random(0)
for ax, class_dir in zip(axes.flat, class_dirs):
    imgs = list(class_dir.glob('*.ppm')) + list(class_dir.glob('*.png'))
    if not imgs:
        ax.axis('off'); continue
    img = Image.open(rng.choice(imgs)).resize((64, 64))
    ax.imshow(img)
    ax.set_title(f'class {int(class_dir.name)}', fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(class_dirs):]:
    ax.axis('off')
fig.tight_layout()
plt.show()

In [ ]:
counts = pd.Series({int(d.name): sum(1 for _ in d.iterdir()) for d in class_dirs}).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(14, 4.5))
sns.barplot(x=counts.index, y=counts.values, color='#3b7dd8', ax=ax)
ax.set(title='GTSRB — samples per class (training split)',
       xlabel='Class id', ylabel='Samples')
ax.tick_params(axis='x', rotation=90, labelsize=8)
fig.tight_layout(); plt.show()
print('imbalance ratio (max/min):', int(counts.max() / counts.min()))

## 2. Training curves

Each training run writes a `training_history.json` under
`reports/metrics/<model>/`. We load the three of them and plot them
side by side.

In [ ]:
import json

MODELS = ['traffic_sign_net', 'traffic_sign_net_stn', 'deep_traffic_net']
REPORTS = Path('../reports/metrics')

histories = {}
for m in MODELS:
    p = REPORTS / m / 'training_history.json'
    if p.is_file():
        histories[m] = json.loads(p.read_text())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for m, h in histories.items():
    epochs = [r['epoch'] for r in h['records']]
    axes[0].plot(epochs, [r['val_acc'] for r in h['records']], marker='o', ms=3, label=m)
    axes[1].plot(epochs, [r['val_loss'] for r in h['records']], marker='o', ms=3, label=m)
axes[0].set(title='Validation accuracy', xlabel='Epoch', ylabel='Accuracy'); axes[0].legend()
axes[1].set(title='Validation loss', xlabel='Epoch', ylabel='Cross-entropy'); axes[1].legend()
fig.tight_layout(); plt.show()

## 3. Confusion matrices & per-class reports

In [ ]:
from IPython.display import Image as NBImage, display

for m in MODELS:
    p = Path(f'../reports/figures/{m}/confusion_matrix.png')
    if p.is_file():
        print(m)
        display(NBImage(filename=str(p)))

## 4. Comparison table

The comparison table is built from each model's `test_metrics.json`
object — meaning every column is sourced from the same object and the
same test set for every model, unlike the mixed train/val/test metrics
in the original notebook.

In [ ]:
rows = []
for m in MODELS:
    p = REPORTS / m / 'test_metrics.json'
    if not p.is_file():
        continue
    r = json.loads(p.read_text())
    rows.append({
        'Model': r['model'],
        'Test Accuracy': r['accuracy'],
        'Top-5 Accuracy': r['top5_accuracy'],
        'Test Loss': r['loss'],
        'MCC': r['mcc'],
        'Kappa': r['kappa'],
        'N (test)': r['num_samples'],
    })
df = pd.DataFrame(rows)
df.style.format({'Test Accuracy': '{:.2%}', 'Top-5 Accuracy': '{:.2%}',
                 'Test Loss': '{:.4f}', 'MCC': '{:.4f}', 'Kappa': '{:.4f}'})

## 5. Single-image inference

Pick any image, load the best checkpoint of the strongest model, and
render the top-5 predictions.

In [ ]:
from traffic_signs.inference.predict import load_predictor

MODEL = 'deep_traffic_net'
predictor = load_predictor(
    checkpoint_path=Path(f'../checkpoints/{MODEL}/best.pt'),
    model_name=MODEL,
    num_classes=43,
    image_size=48,
    class_names_file=Path(f'../checkpoints/{MODEL}/class_names.json'),
    device='cpu',
)

sample_path = next((TRAIN_DIR / '00014').glob('*'))
pred = predictor.predict(sample_path, top_k=5)
print(f'Predicted: class {pred.class_id} ({pred.class_name}) — {pred.probability:.1%}')
for name, prob in pred.top_k:
    print(f'  {name:>8s}  {prob:.2%}')